# 📘 Agentic Architectures 18: Plan-Execute-Observe-Replan (Dynamic Replanning)

In this notebook we explore the **Plan-Execute-Observe-Replan** architecture — a powerful pattern that takes agent adaptability to a new level. Unlike the PEV (Plan-Execute-Verify) architecture, where the Verifier simply checks for success/failure, here we add an **Observer**, which performs deep analysis of results, and a **Replanner**, which is capable of fully reconsidering the strategy.

The key difference in this architecture is the ability not just to repeat a failed step, but to **completely rethink the approach** to solving the task based on the observations gathered. This is especially valuable when working with unreliable tools, dynamically changing data, or tasks where the initial plan turns out to be suboptimal.

To demonstrate this, we'll build a set of "flaky" tools that simulate real-world problems: partial data, timeouts, stale information. We'll then compare the behavior of a baseline PEV agent against our Replan agent.

### Definition
The **Plan-Execute-Observe-Replan** architecture is an extended agent control loop, where every action is not only checked for success but also analyzed for the insights it produced. The Observer extracts information that may affect the further strategy, and the Replanner adapts the plan based on these observations.

### High-level workflow

1.  **Plan:** The agent creates an initial plan to achieve the goal.
2.  **Execute:** The Executor performs the next step of the plan.
3.  **Observe:** The Observer analyzes the result:
    *   Extracts key facts and data
    *   Determines whether the result matches expectations
    *   Identifies new information that affects the plan
4.  **Replan:** Based on the observations, a decision is made:
    *   **Continue** the current plan
    *   **Modify** the plan (add/remove steps)
    *   **Completely reconsider** the strategy

### When to use it / Applications
*   **Dynamic environments:** When data or conditions change during execution.
*   **Exploratory tasks:** When it's not known in advance which path will lead to success.
*   **Unreliable tools:** When working with APIs that may return incomplete or stale data.
*   **Complex multi-step tasks:** Where the result of one step significantly affects the next.

### Strengths and weaknesses
*   **Strengths:**
    *   **High adaptability:** The ability to change strategy "on the fly."
    *   **Deep analysis:** The Observer extracts more information than a simple check.
    *   **Resilience to uncertainty:** Works even when the initial plan is suboptimal.
*   **Weaknesses:**
    *   **Increased cost:** More LLM calls for analysis and replanning.
    *   **Risk of infinite loops:** Limits on the number of replans are necessary.
    *   **Debugging complexity:** Dynamically changing plans are harder to trace.

## Phase 0: Foundation and Setup

We'll start by installing our libraries and configuring the API keys.

### Step 0.1: Installing libraries

**What we'll do:**
We'll install the necessary libraries: `langchain`, `langgraph` for the agent logic, `rich` for pretty output, and `langchain-nebius` for access to the models.

In [ ]:
# !pip install -q -U langchain-nebius langchain langgraph rich python-dotenv langchain-tavily

### Step 0.2: Imports and environment setup

**What we'll do:**
1. Import the classes for graphs, models, and prompts.
2. Load the API keys from the `.env` file.
3. Configure LangSmith tracing for debugging.
4. Initialize the Rich Console for colored log output.

In [ ]:
import os
import json
import random
import time
from typing import List, Annotated, TypedDict, Optional, Literal, Dict, Any
from dotenv import load_dotenv

# Pydantic for data modeling
from pydantic import BaseModel, Field

# LangChain components
from langchain_nebius import ChatNebius
from langchain_community.tools.tavily_search import TavilySearchResults as TavilySearch
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate

# LangGraph components
from langgraph.graph import StateGraph, END
from langgraph.graph.message import AnyMessage, add_messages

# For nice output
from rich.console import Console
from rich.markdown import Markdown
from rich.panel import Panel
from rich.table import Table

from notebook_utils import setup_environment

# --- Configure API keys and tracing ---
console = setup_environment(
    project_name="Agentic Architecture - Replan Dynamic (Nebius)",
    required_keys=["NEBIUS_API_KEY", "LANGCHAIN_API_KEY", "TAVILY_API_KEY"]
)

### Step 0.3: Defining the "Flaky" Tools

**What we'll do:**
We'll create special versions of tools that intentionally misbehave. This is necessary to test the agent's ability to recover.

**Types of failures:**
1. **`flaky_search_tool`**:
   - Sometimes returns partial data.
   - Sometimes raises a timeout error.
   - May return stale data.
2. **`flaky_data_tool`**:
   - May return data in the wrong format (XML instead of JSON).
3. **`flaky_calculate_tool`**:
   - Requires clarification if there isn't enough data for the calculation.

In [ ]:
console = Console()
llm = ChatNebius(model="meta-llama/Meta-Llama-3.1-8B-Instruct", temperature=0)

# Call counter to simulate different behaviors
tool_call_counts = {"search": 0, "data": 0, "calculate": 0}

def flaky_search_tool(query: str) -> str:
    """A search tool that sometimes returns incomplete results."""
    tool_call_counts["search"] += 1
    console.print(f"--- TOOL [flaky_search]: Searching '{query}'... (call #{tool_call_counts['search']}) ---")

    # Simulate various failures
    if "competitor" in query.lower() and tool_call_counts["search"] == 1:
        console.print("--- TOOL: [bold yellow]Returning incomplete data![/bold yellow] ---")
        return "PARTIAL_DATA: Found information about 2 of 5 competitors. Main competitor: CompanyX with 30% market share. [Data incomplete due to API limitations]"

    if "revenue" in query.lower() and tool_call_counts["search"] <= 2:
        console.print("--- TOOL: [bold red]API timeout![/bold red] ---")
        return "ERROR: Request timeout after 30s. The financial data service is temporarily unavailable. Suggest trying alternative source."

    if "market" in query.lower() and tool_call_counts["search"] == 1:
        console.print("--- TOOL: [bold yellow]Stale data![/bold yellow] ---")
        return "STALE_DATA (from 2023-Q2): Market size was $50B. WARNING: This data is 18 months old. Current estimates may differ by 20-30%."

    # Successful result via Tavily
    try:
        result = TavilySearch(max_results=2).invoke(query)
        if isinstance(result, (dict, list)):
            return json.dumps(result, indent=2)
        return str(result)
    except Exception as e:
        return f"Search completed. Found relevant information about: {query}"

def flaky_data_tool(data_type: str) -> str:
    """A data-retrieval tool with various types of failures."""
    tool_call_counts["data"] += 1
    console.print(f"--- TOOL [flaky_data]: Requesting data '{data_type}'... (call #{tool_call_counts['data']}) ---")

    if "employee" in data_type.lower():
        if tool_call_counts["data"] == 1:
            console.print("--- TOOL: [bold red]Data format error![/bold red] ---")
            return "FORMAT_ERROR: Expected JSON but received XML. Raw data: <employees><count>unknown</count></employees>"
        else:
            return "Employee count: 15,000 (as of Q4 2024)"

    if "financial" in data_type.lower():
        return json.dumps({
            "revenue": "$25.5B",
            "growth": "15% YoY",
            "profit_margin": "22%"
        })

    return f"Data retrieved for: {data_type}"

def flaky_calculate_tool(expression: str) -> str:
    """A calculator that sometimes requires clarification."""
    tool_call_counts["calculate"] += 1
    console.print(f"--- TOOL [flaky_calculate]: Calculating '{expression}'... ---")

    if "per employee" in expression.lower() and "revenue" not in expression.lower():
        return "CLARIFICATION_NEEDED: Cannot calculate 'per employee' metric. Missing revenue or other base value. Please provide the numerator."

    # Simple calculator
    try:
        # Extract numbers for a simple calculation
        numbers = [float(x) for x in expression.split() if x.replace('.', '').replace(',', '').isdigit()]
        if len(numbers) >= 2:
            return f"Calculation result: {numbers[0] / numbers[1]:.2f}"
    except:
        pass

    return f"Processed calculation: {expression}"

# Reset counters
def reset_tool_counters():
    global tool_call_counts
    tool_call_counts = {"search": 0, "data": 0, "calculate": 0}

print("Flaky tools defined.")

## Phase 1: Baseline — the PEV agent

First, we'll build a simplified PEV (Plan-Execute-Verify) agent for comparison. This agent has simple verification logic: it checks whether the result contains an error, and either continues or retries the same step.

### Step 1.1: Implementing the PEV structure

**What we'll do:**
1. Define the `PEVState` state to hold the plan and intermediate data.
2. Create the graph nodes:
   - **`pev_planner_node`**: Creates a linear plan of steps.
   - **`pev_executor_node`**: Executes the current step.
   - **`pev_verifier_node`**: Checks whether execution succeeded (Success/Retry).
   - **`pev_router`**: Determines the next step (Retry, Execute Next, or Finish).
3. Assemble the `pev_agent` graph.

In [ ]:
# Pydantic models for structured output
class Plan(BaseModel):
    steps: List[str] = Field(description="List of steps to execute")

class VerificationResult(BaseModel):
    is_successful: bool = Field(description="Whether the step succeeded")
    should_retry: bool = Field(description="Whether the step should be retried")
    reasoning: str = Field(description="Justification for the decision")

class PEVState(TypedDict):
    user_request: str
    plan: Optional[List[str]]
    current_step_index: int
    last_result: Optional[str]
    collected_data: List[str]
    final_answer: Optional[str]
    retry_count: int
    max_retries: int

def pev_planner_node(state: PEVState):
    console.print("--- [PEV] PLANNER: Creating the plan... ---")
    planner_llm = llm.with_structured_output(Plan)

    prompt = f"""
    Create a plan to answer this request: "{state['user_request']}"

    Available tools:
    - flaky_search_tool(query): Search for information
    - flaky_data_tool(data_type): Get specific data
    - flaky_calculate_tool(expression): Perform calculations

    Return a list of 3-5 specific steps using these tools.
    """

    plan = planner_llm.invoke(prompt)
    console.print(f"[cyan]Plan: {plan.steps}[/cyan]")
    return {"plan": plan.steps, "current_step_index": 0, "collected_data": []}

def pev_executor_node(state: PEVState):
    step_idx = state["current_step_index"]
    step = state["plan"][step_idx]
    console.print(f"--- [PEV] EXECUTOR: Executing step {step_idx + 1}: '{step}' ---")

    # Determine which tool to use
    if "search" in step.lower() or "find" in step.lower():
        result = flaky_search_tool(step)
    elif "data" in step.lower() or "get" in step.lower():
        result = flaky_data_tool(step)
    elif "calcul" in step.lower() or "compute" in step.lower():
        result = flaky_calculate_tool(step)
    else:
        result = flaky_search_tool(step)

    return {"last_result": result}

def pev_verifier_node(state: PEVState):
    console.print("--- [PEV] VERIFIER: Checking the result... ---")
    verifier_llm = llm.with_structured_output(VerificationResult)

    prompt = f"""
    Verify this tool result:
    Step: {state['plan'][state['current_step_index']]}
    Result: {state['last_result']}

    Determine if the step was successful or if it should be retried.
    Look for ERROR, TIMEOUT, PARTIAL_DATA, FORMAT_ERROR keywords.
    """

    verification = verifier_llm.invoke(prompt)
    console.print(f"--- [PEV] VERIFIER: {'✓ Success' if verification.is_successful else '✗ Retry needed'} ---")

    if verification.is_successful:
        new_data = state["collected_data"] + [state["last_result"]]
        return {
            "collected_data": new_data,
            "current_step_index": state["current_step_index"] + 1,
            "retry_count": 0
        }
    else:
        return {"retry_count": state["retry_count"] + 1}

def pev_synthesizer_node(state: PEVState):
    console.print("--- [PEV] SYNTHESIZER: Generating the answer... ---")

    context = "\n".join(state["collected_data"])
    prompt = f"""
    Synthesize an answer for: "{state['user_request']}"

    Collected data:
    {context}

    Provide a comprehensive answer based on the available data.
    Note any limitations if data was incomplete.
    """

    answer = llm.invoke(prompt).content
    return {"final_answer": answer}

def pev_router(state: PEVState):
    # Check whether the retry limit has been exceeded
    if state["retry_count"] >= state["max_retries"]:
        console.print("--- [PEV] ROUTER: Retry limit reached, moving to synthesis... ---")
        # Add partial data and move on
        return "skip_step"

    if state["retry_count"] > 0:
        console.print(f"--- [PEV] ROUTER: Retrying step (attempt {state['retry_count'] + 1})... ---")
        return "retry"

    if state["current_step_index"] >= len(state["plan"]):
        console.print("--- [PEV] ROUTER: Plan complete, moving to synthesis ---")
        return "synthesize"

    console.print("--- [PEV] ROUTER: Continuing execution ---")
    return "continue"

def pev_skip_step_node(state: PEVState):
    console.print("--- [PEV] SKIP: Skipping step with partial data ---")
    new_data = state["collected_data"] + [f"[SKIPPED with partial data: {state['last_result']}]"]
    return {
        "collected_data": new_data,
        "current_step_index": state["current_step_index"] + 1,
        "retry_count": 0
    }

# Build the PEV graph
pev_graph = StateGraph(PEVState)
pev_graph.add_node("plan", pev_planner_node)
pev_graph.add_node("execute", pev_executor_node)
pev_graph.add_node("verify", pev_verifier_node)
pev_graph.add_node("skip_step", pev_skip_step_node)
pev_graph.add_node("synthesize", pev_synthesizer_node)

pev_graph.set_entry_point("plan")
pev_graph.add_edge("plan", "execute")
pev_graph.add_edge("execute", "verify")
pev_graph.add_conditional_edges("verify", pev_router, {
    "continue": "execute",
    "retry": "execute",
    "skip_step": "skip_step",
    "synthesize": "synthesize"
})
pev_graph.add_conditional_edges("skip_step", lambda s: "synthesize" if s["current_step_index"] >= len(s["plan"]) else "execute")
pev_graph.add_edge("synthesize", END)

pev_agent = pev_graph.compile()
print("Baseline PEV agent compiled.")

## Phase 2: The advanced approach — the Replan Dynamic agent

Now we'll build a full-fledged Replan agent with:
- **Observer** — deep analysis of results, extracting insights
- **Replanner** — intelligent replanning based on observations

### Step 2.1: Implementing the Replan agent's nodes

**What we'll do:**
1. Create an `Observation` model for deep analysis of results (facts, issues, suggestions).
2. Create a `ReplanDecision` model for choosing a strategy (Continue, Modify, Full Replan).
3. Implement the nodes:
   - **`replan_observer_node`**: Analyzes the tool output, identifies "flaky" patterns.
   - **`replan_replanner_node`**: Decides whether to change the plan based on observations.
   - **`replan_executor_node`**: Executes steps and stores the full history (step + result).

In [ ]:
# Pydantic models for the Replan agent
class Observation(BaseModel):
    status: Literal["success", "partial", "failure", "needs_alternative"] = Field(
        description="Status of the step's execution"
    )
    extracted_facts: List[str] = Field(
        description="Facts extracted from the result"
    )
    issues_found: List[str] = Field(
        description="Issues found"
    )
    suggestions: List[str] = Field(
        description="Suggestions for improving the plan"
    )

class ReplanDecision(BaseModel):
    action: Literal["continue", "modify_plan", "full_replan", "synthesize"] = Field(
        description="Decision about further actions"
    )
    new_steps: Optional[List[str]] = Field(
        default=None,
        description="New or modified steps (if applicable)"
    )
    reasoning: str = Field(
        description="Justification for the decision"
    )

class ReplanState(TypedDict):
    user_request: str
    current_plan: List[str]
    current_step_index: int
    execution_history: List[Dict[str, Any]]  # {step, result, observation}
    observations: List[str]
    extracted_facts: List[str]
    final_answer: Optional[str]
    replan_count: int
    max_replans: int

def replan_planner_node(state: ReplanState):
    console.print("--- [REPLAN] PLANNER: Creating the initial plan... ---")
    planner_llm = llm.with_structured_output(Plan)

    # Take previous attempts into account, if there were any
    history_context = ""
    if state.get("execution_history"):
        history_context = f"\nPrevious attempts and their results:\n{json.dumps(state['execution_history'], indent=2)}"

    prompt = f"""
    Create a plan to answer: "{state['user_request']}"
    {history_context}

    Available tools:
    - flaky_search_tool(query): Search for information (may return partial data)
    - flaky_data_tool(data_type): Get specific data (may have format issues)
    - flaky_calculate_tool(expression): Perform calculations

    Create a robust plan with 3-5 steps. Consider using alternative approaches if primary ones might fail.
    """

    plan = planner_llm.invoke(prompt)
    console.print(Panel("\n".join([f"{i+1}. {s}" for i, s in enumerate(plan.steps)]), title="Plan", border_style="cyan"))

    return {
        "current_plan": plan.steps,
        "current_step_index": 0,
        "execution_history": state.get("execution_history", []),
        "extracted_facts": state.get("extracted_facts", [])
    }

def replan_executor_node(state: ReplanState):
    step_idx = state["current_step_index"]
    step = state["current_plan"][step_idx]
    console.print(f"--- [REPLAN] EXECUTOR: Step {step_idx + 1}/{len(state['current_plan'])}: '{step}' ---")

    # Tool selection
    if "search" in step.lower() or "find" in step.lower() or "look" in step.lower():
        result = flaky_search_tool(step)
    elif "data" in step.lower() or "get" in step.lower() or "retrieve" in step.lower():
        result = flaky_data_tool(step)
    elif "calcul" in step.lower() or "compute" in step.lower():
        result = flaky_calculate_tool(step)
    else:
        result = flaky_search_tool(step)

    # Save to history
    history_entry = {"step": step, "step_index": step_idx, "result": result}
    new_history = state["execution_history"] + [history_entry]

    return {"execution_history": new_history}

def replan_observer_node(state: ReplanState):
    console.print("--- [REPLAN] OBSERVER: Analyzing the result... ---")
    observer_llm = llm.with_structured_output(Observation)

    last_execution = state["execution_history"][-1]

    prompt = f"""
    Analyze this tool execution result:

    Step: {last_execution['step']}
    Result: {last_execution['result']}

    Task context: {state['user_request']}

    Analyze the result and provide:
    1. Status: success (got what we needed), partial (got some data), failure (error), needs_alternative (need different approach)
    2. Any facts or data that can be extracted from this result
    3. Any issues that were encountered
    4. Suggestions for improving the plan if needed

    Look for keywords like ERROR, PARTIAL_DATA, STALE_DATA, TIMEOUT, FORMAT_ERROR, CLARIFICATION_NEEDED.
    """

    try:
        observation = observer_llm.invoke(prompt)
    except Exception as e:
        # Fallback if parsing failed
        observation = Observation(
            status="partial",
            extracted_facts=[last_execution['result'][:200]],
            issues_found=["Parsing error in observation"],
            suggestions=[]
        )

    status_emoji = {"success": "✅", "partial": "⚠️", "failure": "❌", "needs_alternative": "🔄"}
    console.print(f"--- [REPLAN] OBSERVER: Status: {status_emoji.get(observation.status, '?')} {observation.status} ---")

    if observation.extracted_facts:
        console.print(f"[green]Facts extracted: {len(observation.extracted_facts)}[/green]")
    if observation.issues_found:
        console.print(f"[yellow]Issues: {observation.issues_found}[/yellow]")

    # Update state
    new_facts = state["extracted_facts"] + observation.extracted_facts
    new_observations = state.get("observations", []) + [observation.model_dump_json()]

    # Update history with the observation
    updated_history = state["execution_history"].copy()
    updated_history[-1]["observation"] = observation.model_dump()

    return {
        "execution_history": updated_history,
        "observations": new_observations,
        "extracted_facts": new_facts
    }

def replan_replanner_node(state: ReplanState):
    console.print("--- [REPLAN] REPLANNER: Making a decision... ---")
    replanner_llm = llm.with_structured_output(ReplanDecision)

    last_observation = state["execution_history"][-1].get("observation", {})
    remaining_steps = state["current_plan"][state["current_step_index"] + 1:]

    prompt = f"""
    Decide on the next action based on the execution history.

    User request: {state['user_request']}

    Current step completed: {state['current_step_index'] + 1}/{len(state['current_plan'])}
    Last observation: {json.dumps(last_observation)}

    Remaining steps in plan: {remaining_steps}
    Extracted facts so far: {state['extracted_facts']}

    Replan count: {state['replan_count']}/{state['max_replans']}

    Choose action:
    - "continue": Move to next step in current plan
    - "modify_plan": Modify remaining steps based on observations
    - "full_replan": Create entirely new plan (use sparingly, counts toward limit)
    - "synthesize": Enough data collected, generate final answer

    If modifying or replanning, provide the new steps.
    """

    try:
        decision = replanner_llm.invoke(prompt)
    except Exception as e:
        # Fallback
        decision = ReplanDecision(
            action="continue",
            reasoning="Fallback to continue"
        )

    action_emoji = {"continue": "➡️", "modify_plan": "📝", "full_replan": "🔄", "synthesize": "📊"}
    console.print(f"--- [REPLAN] REPLANNER: {action_emoji.get(decision.action, '?')} {decision.action} ---")
    console.print(f"[dim]Reason: {decision.reasoning}[/dim]")

    if decision.action == "continue":
        return {"current_step_index": state["current_step_index"] + 1}

    elif decision.action == "modify_plan":
        if decision.new_steps:
            # Replace the remaining steps
            new_plan = state["current_plan"][:state["current_step_index"] + 1] + decision.new_steps
            console.print(Panel("\n".join([f"{i+1}. {s}" for i, s in enumerate(new_plan)]), title="Modified plan", border_style="yellow"))
            return {
                "current_plan": new_plan,
                "current_step_index": state["current_step_index"] + 1
            }
        return {"current_step_index": state["current_step_index"] + 1}

    elif decision.action == "full_replan":
        return {
            "replan_count": state["replan_count"] + 1,
            "current_step_index": 0
        }

    else:  # synthesize
        return {}  # Handled by router

def replan_synthesizer_node(state: ReplanState):
    console.print("--- [REPLAN] SYNTHESIZER: Generating the answer... ---")

    facts = "\n".join([f"- {fact}" for fact in state["extracted_facts"]])

    prompt = f"""
    Generate a comprehensive answer.

    User request: {state['user_request']}

    Extracted facts:
    {facts}

    Execution summary:
    - Total steps executed: {len(state['execution_history'])}
    - Replanning events: {state['replan_count']}

    Provide a comprehensive answer. If some data was incomplete or had issues, acknowledge the limitations.
    """

    answer = llm.invoke(prompt).content
    return {"final_answer": answer}

def replan_router(state: ReplanState):
    # Check the replanner's last decision
    last_execution = state["execution_history"][-1] if state["execution_history"] else {}
    last_observation = last_execution.get("observation", {})

    # Check the replan limit
    if state["replan_count"] >= state["max_replans"]:
        console.print("--- [REPLAN] ROUTER: Replan limit reached, finalizing ---")
        return "synthesize"

    # If a full replan is needed
    if state["current_step_index"] == 0 and state["replan_count"] > 0:
        return "plan"

    # If all steps have been executed
    if state["current_step_index"] >= len(state["current_plan"]):
        return "synthesize"

    return "execute"

# Build the Replan graph
replan_graph = StateGraph(ReplanState)
replan_graph.add_node("plan", replan_planner_node)
replan_graph.add_node("execute", replan_executor_node)
replan_graph.add_node("observe", replan_observer_node)
replan_graph.add_node("replan", replan_replanner_node)
replan_graph.add_node("synthesize", replan_synthesizer_node)

replan_graph.set_entry_point("plan")
replan_graph.add_edge("plan", "execute")
replan_graph.add_edge("execute", "observe")
replan_graph.add_edge("observe", "replan")
replan_graph.add_conditional_edges("replan", replan_router, {
    "plan": "plan",
    "execute": "execute",
    "synthesize": "synthesize"
})
replan_graph.add_edge("synthesize", END)

replan_agent = replan_graph.compile()
print("Replan Dynamic agent compiled.")

## Phase 3: Comparing the agents

Now let's run both agents on the same complex task, one that requires adapting to tool failures.

### Step 3.1: Testing and Comparison

**What we'll do:**
1. Define a complex, multi-step query that is guaranteed to trigger failures.
2. Run the PEV agent and see how it copes (we expect it to retry steps).
3. Run the Replan agent and observe the strategy adapting.
4. Use `ComparisonResult` to automatically score the quality of both agents' work.

In [ ]:
test_query = """Analyze TechCorp company:
1. Find their main competitors and market share
2. Get their revenue data for the last fiscal year
3. Calculate revenue per employee
4. Provide an overall assessment"""

console.print(Panel(test_query, title="Test query", border_style="magenta"))

In [ ]:
console.print("\n" + "="*60)
console.print("[bold blue]Test 1: Baseline PEV agent[/bold blue]")
console.print("="*60 + "\n")

reset_tool_counters()

pev_result = pev_agent.invoke({
    "user_request": test_query,
    "plan": None,
    "current_step_index": 0,
    "last_result": None,
    "collected_data": [],
    "final_answer": None,
    "retry_count": 0,
    "max_retries": 2
}, {"recursion_limit": 30})

console.print("\n[bold]PEV agent result:[/bold]")
console.print(Panel(Markdown(pev_result["final_answer"]), border_style="blue"))

In [ ]:
console.print("\n" + "="*60)
console.print("[bold green]Test 2: Replan Dynamic agent[/bold green]")
console.print("="*60 + "\n")

reset_tool_counters()

replan_result = replan_agent.invoke({
    "user_request": test_query,
    "current_plan": [],
    "current_step_index": 0,
    "execution_history": [],
    "observations": [],
    "extracted_facts": [],
    "final_answer": None,
    "replan_count": 0,
    "max_replans": 3
}, {"recursion_limit": 40})

console.print("\n[bold]Replan agent result:[/bold]")
console.print(Panel(Markdown(replan_result["final_answer"]), border_style="green"))

### Comparative analysis

In [ ]:
# Evaluation via LLM-as-Judge
class ComparisonResult(BaseModel):
    pev_score: int = Field(description="PEV agent score (1-10)")
    replan_score: int = Field(description="Replan agent score (1-10)")
    pev_strengths: List[str] = Field(description="PEV agent's strengths")
    replan_strengths: List[str] = Field(description="Replan agent's strengths")
    winner: str = Field(description="Winner: PEV or Replan")
    justification: str = Field(description="Justification")

judge_llm = llm.with_structured_output(ComparisonResult)

judge_prompt = f"""
Compare these two agent outputs for the same task.
Task: {test_query}
PEV Agent Output: {pev_result['final_answer']}
Replan Agent Output: {replan_result['final_answer']}
"""

comparison = judge_llm.invoke(judge_prompt)

console.print(Panel(
    f"[bold]Winner:[/bold] {comparison.winner}\n\n"
    f"[bold]PEV Score:[/bold] {comparison.pev_score}/10\n"
    f"[bold]Replan Score:[/bold] {comparison.replan_score}/10\n\n"
    f"[bold]Justification:[/bold] {comparison.justification}",
    title="LLM Judge Comparison",
    border_style="gold1"
))